<a href="https://colab.research.google.com/github/mt508/machine-learning-/blob/main/ecg_perdiction_project_git.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
data=files.upload()

In [ ]:
!unzip ecg_dataset.zip

In [ ]:
import os
import shutil
import random
from sklearn.model_selection import StratifiedShuffleSplit


In [ ]:
dataset_dir = "/content/ecg_dataset"
classes = os.listdir(dataset_dir)

image_paths = []
labels = []

for label_idx, class_name in enumerate(classes):
    class_dir = os.path.join(dataset_dir, class_name)
    for img in os.listdir(class_dir):
        if img.lower().endswith((".png", ".jpg", ".jpeg")):
            image_paths.append(os.path.join(class_dir, img))
            labels.append(label_idx)


In [ ]:
sss1 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

for train_idx, temp_idx in sss1.split(image_paths, labels):
    train_paths = [image_paths[i] for i in train_idx]
    train_labels = [labels[i] for i in train_idx]

    temp_paths = [image_paths[i] for i in temp_idx]
    temp_labels = [labels[i] for i in temp_idx]


In [ ]:
sss2 = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

for val_idx, test_idx in sss2.split(temp_paths, temp_labels):
    val_paths = [temp_paths[i] for i in val_idx]
    test_paths = [temp_paths[i] for i in test_idx]


In [ ]:
output_dir = "ecg_split"

splits = ["train", "val", "test"]

for split in splits:
    for class_name in classes:
        os.makedirs(os.path.join(output_dir, split, class_name), exist_ok=True)


In [ ]:
def copy_images(image_list, split_name):
    for img_path in image_list:
        class_name = os.path.basename(os.path.dirname(img_path))
        dest = os.path.join(output_dir, split_name, class_name)
        shutil.copy(img_path, dest)

copy_images(train_paths, "train")
copy_images(val_paths, "val")
copy_images(test_paths, "test")


In [ ]:
from collections import Counter

print("Train:", Counter(train_labels))
print("Val  :", Counter([labels[image_paths.index(p)] for p in val_paths]))
print("Test :", Counter([labels[image_paths.index(p)] for p in test_paths]))


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG_SIZE = 224
BATCH_SIZE = 16

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(3),
    transforms.ToTensor()
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder("/content/ecg_split/train", transform=train_tfms)
val_ds   = datasets.ImageFolder("/content/ecg_split/val", transform=val_tfms)
test_ds  = datasets.ImageFolder("/content/ecg_split/test", transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


resnet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# train_loader, val_loader already defined
num_classes = 3


In [ ]:
model = models.resnet18(pretrained=True)

# Replace final FC layer
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)


In [ ]:
import numpy as np

# From your Train Counter
class_counts = np.array([192, 137, 236])  # [Abnormal, MI, Normal]

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.3
)


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
def validate(model, loader):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
EPOCHS = 20
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    scheduler.step(val_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "resnet18_ecg_best.pth")


In [ ]:
torch.save(model.state_dict(), "resnet18_ecg_best.pth")


In [ ]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Abnormal", "History_of_MI", "Normal"]
))


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1).cpu().numpy()
        all_probs.extend(probs)

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

roc_auc = roc_auc_score(
    all_labels,
    all_probs,
    multi_class="ovr"
)

print(f"ROC-AUC (OvR): {roc_auc:.4f}")


In [ ]:
print(test_loader.dataset.root)


In [ ]:
roc_auc_score(all_labels, all_probs, multi_class="ovr")


In [ ]:
print(all_probs.shape)
print(all_probs[:5])
print(all_labels[:5])


tiny vit

In [ ]:
!pip install timm


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from tqdm import tqdm
import numpy as np


In [ ]:
model = timm.create_model(
    "tiny_vit_21m_224",
    pretrained=True,
    num_classes=num_classes
)

model = model.to(device)


In [ ]:
# From your Train Counter
class_counts = np.array([192, 137, 236])  # [Abnormal, MI, Normal]

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

weights = torch.tensor(class_weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)


In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)


In [ ]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
def validate(model, loader):
    model.eval()
    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), correct / total


In [ ]:
EPOCHS = 20
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = validate(model, val_loader)

    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "tinyvit_ecg_best.pth")


In [ ]:
model.load_state_dict(torch.load("tinyvit_ecg_best.pth"))
model.eval()


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize

all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(probs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)


In [ ]:
test_acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Abnormal", "MI", "Normal"],
            yticklabels=["Abnormal", "MI", "Normal"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# One-vs-Rest binarization
y_true_bin = label_binarize(all_labels, classes=[0, 1, 2])

roc_auc = roc_auc_score(
    y_true_bin,
    all_probs,
    multi_class="ovr",
    average="macro"
)

print(f"ROC-AUC (OvR): {roc_auc:.4f}")


prediction

In [ ]:
from PIL import Image
from torchvision import transforms

img_path = "/content/Screenshot 2026-01-09 004320.png"  # replace with your file
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

img = Image.open(img_path).convert("RGB")
img_tensor = transform(img).unsqueeze(0).to(device)


In [ ]:
model.load_state_dict(torch.load("tinyvit_ecg_best.pth"))
model.eval()


In [ ]:
with torch.no_grad():
    outputs = model(img_tensor)
    probs = torch.softmax(outputs, dim=1)
    pred = torch.argmax(probs, dim=1).item()

print("Predicted class:", pred)
print("Class probabilities:", probs.cpu().numpy())


In [ ]:
from torchvision.datasets import ImageFolder

print(train_dataset.class_to_idx)
